# Session 4b: LangGraph Stateful Orchestration

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Send

print("LangGraph ready")

## 1. Simple StateGraph

In [ ]:
class SimpleState(TypedDict):
    query: str
    result: str

def process(state: SimpleState):
    return {"result": f"Processed: {state['query']}"}

builder = StateGraph(SimpleState)
builder.add_node("process", process)
builder.add_edge(START, "process")
builder.add_edge("process", END)
graph = builder.compile()

result = graph.invoke({"query": "Analyze AAPL"})
print(result)

## 2. Parallel Dispatch with Send()

In [ ]:
class ResearchState(TypedDict):
    query: str
    market_data: str
    sentiment: str
    macro: str
    synthesis: str

async def research_market(state: ResearchState):
    return {"market_data": f"{state['query']}: P/E 28, Revenue $120B"}

async def research_sentiment(state: ResearchState):
    return {"sentiment": f"{state['query']}: Bullish (score 0.75)"}

async def research_macro(state: ResearchState):
    return {"macro": "GDP 2.8%, CPI 3.1%, Fed 5.5%"}

async def synthesize(state: ResearchState):
    combined = f"Market: {state['market_data']}\nSentiment: {state['sentiment']}\nMacro: {state['macro']}"
    return {"synthesis": combined}

builder = StateGraph(ResearchState)
builder.add_node("market", research_market)
builder.add_node("sentiment", research_sentiment)
builder.add_node("macro", research_macro)
builder.add_node("synthesize", synthesize)

def route_parallel(state: ResearchState):
    return [Send("market", state), Send("sentiment", state), Send("macro", state)]

builder.add_conditional_edges(START, route_parallel)
builder.add_edge("market", "synthesize")
builder.add_edge("sentiment", "synthesize")
builder.add_edge("macro", "synthesize")
builder.add_edge("synthesize", END)

graph = builder.compile()
result = await graph.ainvoke({"query": "AAPL"})
print(result["synthesis"])

## 3. Checkpointing with MemorySaver

In [ ]:
builder = StateGraph(ResearchState)
builder.add_node("process", lambda s: {"synthesis": f"Done: {s['query']}"})
builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "session-1"}}

result = graph.invoke({"query": "Analyze MSFT"}, config)
print(result["synthesis"])

# Resume from checkpoint
state = graph.get_state(config)
print(f"Resumed at step: {state.next}")